# AICAPS transformer training (Colab)
Run one model at a time. The notebook imports the repository's shared `src/` logic and stops cleanly after a five-hour session budget.


In [ ]:
import os,sys,json,time,shutil,subprocess,copy
MODEL=os.environ.get('AICAPS_MODEL','dnabert2'); WINDOW=100
DRIVE_ROOT='/content/drive/MyDrive/aicaps_reproduction' # edit this
REPO_ROOT='/content/aicaps-reproduction'; SESSION_BUDGET_HOURS=5.0
DRY_RUN=os.environ.get('AICAPS_DRY_RUN','0')=='1'; assert MODEL in {'dnabert2','nt'}
# Empty (default) = original random 80/20 split (SPLIT-A / S1), unchanged. Set to a
# manifest path (e.g. 'data/splits/split_manifest_b.json', relative to DRIVE_ROOT, or
# an absolute path) to train against a frozen SPLIT-B/C manifest instead -- sync
# data/splits/*.json to Drive first, same as the parquet under data/processed/.
SPLIT_MANIFEST=os.environ.get('AICAPS_SPLIT_MANIFEST','')
print({'model':MODEL,'window':WINDOW,'budget_hours':SESSION_BUDGET_HOURS,'dry_run':DRY_RUN,'split_manifest':SPLIT_MANIFEST or '(default SPLIT-A)'})


In [ ]:
if not DRY_RUN:
    from google.colab import drive; drive.mount('/content/drive')
    repo_url=os.environ.get('AICAPS_REPO_URL','')
    if repo_url and not os.path.exists(REPO_ROOT): subprocess.run(['git','clone',repo_url,REPO_ROOT],check=True)
else: REPO_ROOT=os.getcwd()
sys.path.insert(0,REPO_ROOT)
if not DRY_RUN: subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.13.0','transformers==4.57.6','accelerate==1.14.0','einops==0.8.2','pandas==2.3.3','numpy==2.3.5','scikit-learn==1.7.2','pyarrow==21.0.0'],check=True)


In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader
from src.training.run_experiment import PairSet,metrics,seed
from src.training.transformer_utils import load_transformer,compute_max_length
from src.data.splits import load_manifest,apply_split
seed(42); DATA=(Path(DRIVE_ROOT)/'data/processed'/f'variants_{WINDOW}bp.parquet') if not DRY_RUN else Path(REPO_ROOT)/'data/processed'/f'variants_{WINDOW}bp.parquet'; OUT=Path(DRIVE_ROOT) if not DRY_RUN else Path(REPO_ROOT)
free_gb=shutil.disk_usage(str(OUT)).free/2**30; print(f'Drive free space before load: {free_gb:.2f} GiB')


In [ ]:
local={'dnabert2':127.40,'nt':242.67}; local_epoch=local[MODEL]/.05
print(f'Local baseline: {local_epoch/60:.1f} minutes/epoch; {local_epoch*10/3600:.1f} hours total')
if MODEL=='nt':
    ratio=local['nt']/local['dnabert2']; log=Path(DRIVE_ROOT)/'results/metrics/dnabert2_100bp_epoch_times.csv'; observed=pd.read_csv(log)['duration_seconds'].mean()/60 if log.exists() else None
    base=observed or local['dnabert2']/.05/60; source='observed DNABERT-2 Colab timing' if observed else 'local fallback (replace after DNABERT-2)'
    print(f'Based on {source}: DNABERT-2 {base:.1f} minutes/epoch and local NT ratio {ratio:.2f}, NT estimated at {base*ratio:.1f} minutes/epoch × 10 = {base*ratio*10/60:.1f} hours total. This will very likely require 2+ sessions.')
    print('NOTE: the local NT ratio above was measured before the tokenizer max_length fix (was 2*WINDOW=200 tokens padded for every NT sequence; now computed from the real k-mer token length, ~40-50). Real NT throughput should come in well under this estimate — treat it as a pessimistic upper bound, not a plan.')


In [ ]:
df=pd.read_parquet(DATA)
# captured before the DRY_RUN row-limiting below, so max_length always reflects the
# true longest ref+mut strings in the full dataset, not just whatever survived a small sample
full_pairs=df.ref_sequence+df.mut_sequence; full_longest=full_pairs.loc[full_pairs.str.len().nlargest(20).index].tolist()
df=df.groupby('label',group_keys=False).head(32) if DRY_RUN else df
if SPLIT_MANIFEST:
    manifest_path=Path(SPLIT_MANIFEST) if os.path.isabs(SPLIT_MANIFEST) else OUT/SPLIT_MANIFEST
    manifest=load_manifest(manifest_path); split_tag='_split'+manifest['split_id'].upper()
    train_df,val_df=apply_split(df,manifest)
else:
    split_tag=''
    train_df,val_df=train_test_split(df,test_size=.2,random_state=42,stratify=df.label)
tokenizer,model=load_transformer('zhihan1996/DNABERT-2-117M' if MODEL=='dnabert2' else 'InstaDeepAI/nucleotide-transformer-v2-100m-multi-species')
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model.to(device); model_bytes=sum(p.numel()*p.element_size() for p in model.parameters()); free_gb=shutil.disk_usage(str(OUT)).free/2**30; projected_gb=model_bytes*3*10*2/2**30
print(f'Loaded model: {model_bytes/2**30:.2f} GiB parameters; projected two-model checkpoint storage: {projected_gb:.2f} GiB; free Drive: {free_gb:.2f} GiB')
if projected_gb>.7*free_gb: print('WARNING: projected storage exceeds 70% of free Drive space')
# DNABERT-2's BPE tokenizer is ~1 token/char, so 2*WINDOW is a safe, near-tight bound.
# NT's tokenizer maps each non-overlapping 6-mer to one token, so 2*WINDOW would pad
# every NT batch to ~5-6x its real length for nothing — compute the true worst case
# from the longest ref+mut strings in the full dataset instead.
if MODEL=='nt':
    maxlen=compute_max_length(tokenizer,full_longest)
else:
    maxlen=2*WINDOW
print('tokenizer max_length used:',maxlen)
train_loader=DataLoader(PairSet(train_df,'transformer',tokenizer,maxlen),batch_size=16,shuffle=True); val_loader=DataLoader(PairSet(val_df,'transformer',tokenizer,maxlen),batch_size=16)
counts=np.bincount(train_df.label,minlength=2); weights=torch.tensor(len(train_df)/(2*np.maximum(counts,1)),dtype=torch.float,device=device); loss_fn=nn.CrossEntropyLoss(weight=weights); optimizer=torch.optim.AdamW(model.parameters(),lr=2e-5,weight_decay=.01); use_amp=device.type=='cuda'; scaler=torch.amp.GradScaler('cuda',enabled=use_amp)
print('device:',device,'train:',len(train_df),'val:',len(val_df),'batch_size:',16)


In [ ]:
CKPT_DIR=OUT/'results/checkpoints'/(MODEL+split_tag.lower()); CKPT_DIR.mkdir(parents=True,exist_ok=True); LOG=OUT/'results/metrics'/f'{MODEL}_{WINDOW}bp{split_tag}_epoch_times.csv'; LOG.parent.mkdir(parents=True,exist_ok=True)
def save_checkpoint(epoch,best_score,best_state):
    p=CKPT_DIR/f'epoch_{epoch:02d}.pt'; torch.save({'epoch':epoch,'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),'best_score':best_score,'best_state':best_state},p); return p
def resume(path):
    ck=torch.load(path,map_location=device); model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer']); scaler.load_state_dict(ck['scaler']); return ck['epoch'],ck['best_score'],ck.get('best_state')
start_epoch=0; best_score=-1.; best_state=None; resume_path=os.environ.get('AICAPS_RESUME_CHECKPOINT','')
if resume_path: start_epoch,best_score,best_state=resume(resume_path); print('resumed after epoch',start_epoch)


In [ ]:
if DRY_RUN:
    model.train(); x,y=next(iter(train_loader)); y=y.to(device); optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=use_amp): loss=loss_fn(model(**{k:v.to(device) for k,v in x.items()}),y)
    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); print('DRY RUN OK: one shared-pipeline training step, loss=',float(loss))
else:
    session_start=time.time()
    for epoch in range(start_epoch,10):
        epoch_start=time.time(); model.train()
        for x,y in train_loader:
            y=y.to(device); optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=use_amp): loss=loss_fn(model(**{k:v.to(device) for k,v in x.items()}),y)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update()
        model.eval(); ys=[]; ps=[]; probs=[]
        with torch.no_grad():
            for x,y in val_loader:
                q=torch.softmax(model(**{k:v.to(device) for k,v in x.items()}),1)[:,1].cpu().numpy(); probs.extend(q); ps.extend((q>=.5).astype(int)); ys.extend(y.numpy())
        m=metrics(ys,ps,probs); score=m['weighted_f1']
        if score>best_score: best_score=score; best_state=copy.deepcopy(model.state_dict())
        ckpt=save_checkpoint(epoch+1,best_score,best_state); end=time.time(); pd.DataFrame([{'model':MODEL,'epoch':epoch+1,'start_time':time.ctime(epoch_start),'end_time':time.ctime(end),'duration_seconds':end-epoch_start}]).to_csv(LOG,index=False,mode='a',header=not LOG.exists()); print(json.dumps({'epoch':epoch+1,'weighted_f1':score,'checkpoint':str(ckpt)}))
        if end-session_start>=SESSION_BUDGET_HOURS*3600: print(f'Stopping early at epoch {epoch+1}/10 to stay within session budget — resume in a new session.'); break


In [ ]:
if not DRY_RUN:
    model.load_state_dict(best_state); model.eval(); ys=[]; ps=[]; probs=[]
    with torch.no_grad():
        for x,y in val_loader:
            q=torch.softmax(model(**{k:v.to(device) for k,v in x.items()}),1)[:,1].cpu().numpy(); probs.extend(q); ps.extend((q>=.5).astype(int)); ys.extend(y.numpy())
    result=metrics(ys,ps,probs); pred=OUT/'results/predictions'; pred.mkdir(parents=True,exist_ok=True); pd.DataFrame({'label':ys,'prediction':ps,'probability_pathogenic':probs}).to_parquet(pred/f'{MODEL}_{WINDOW}bp{split_tag}.parquet',index=False); (OUT/'results/metrics'/f'{MODEL}_{WINDOW}bp{split_tag}_final.json').write_text(json.dumps({'model':MODEL,'window':WINDOW,'metrics':result},indent=2)); print(result)
